# Preparation

Run `pip install -r requirements.txt`, then run this notebook.

# Initial set up

I'm importing packages and defining some globals for the notebook here. I set a random seed for reproducibility.

> **AI usage disclaimer:** I've come to embrace AI-assisted development, the work presented in this notebook has been partially generated using LLMs, however, I've supervised the entire process and ultimately decided which parts to change, update, and customize. This disclaimer is an indicator of my commitment to ethical, responsible, and transparent use of AI.

In [ ]:
import json
import random
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

from datasets import Dataset

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen3VLForConditionalGeneration,
)
from qwen_vl_utils import process_vision_info

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

from trl import SFTConfig, SFTTrainer

In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
OUTPUT_DIR = "qwen3-vl-8b-health-table-extraction-lora"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Overall description

Since we are dealing with complex (and variable) form layouts, we have to represent this data in a structured way. The most natural representation (with many benchmarks available as well) is with standard HTML tables, which allow us to make use of rowspan and colspan to merge cells and create fairly complex tables in a digital form.

**The HTML table** is wrapped in a JSON envelope that carries key/value fields (patient identifiers *redacted for training*, form date, form type):

**AI generated sample**

```json
{
  "form_type": "basic_metabolic_panel",
  "fields": {"specimen_date": "2026-03-14", "ordering_provider": "REDACTED"},
  "table_html": "<table><tr><th>Test</th><th colspan=\"2\">Reference Range</th><th>Result</th></tr>...</table>"
}
```

The model's job is: given the form image, emit exactly this JSON object as a string, while still getting HTML's structural expressiveness for the table body.

# Synthetic training data [AI generated helper]

In [ ]:
def _render_synthetic_lab_form(path: Path, seed: int) -> dict:
    """Render a crude 'lab panel' table into a high-resolution PNG and return
    the matching ground-truth record. This stands in for a real, scanned
    health-form image + a human- or vendor-labeled structured target.

    In production this function is replaced entirely: images come from your
    document ingestion pipeline (scanner/fax/EHR export), and targets come
    from human annotation (e.g., via Label Studio with an HTML-table export
    plugin) or from a trusted upstream structured source used to
    bootstrap weak labels.
    """
    rng = random.Random(seed)
    tests = [
        ("Sodium", "136", "145", "mmol/L"),
        ("Potassium", "3.5", "5.1", "mmol/L"),
        ("Chloride", "98", "107", "mmol/L"),
        ("Glucose", "70", "99", "mg/dL"),
        ("BUN", "7", "20", "mg/dL"),
        ("Creatinine", "0.6", "1.3", "mg/dL"),
    ]

    # High-resolution canvas: scanned health forms are frequently
    # 2000-3000px on the long edge. We deliberately render at high res so
    # the notebook exercises the same resolution regime as real scans.
    width, height = 1700, 2200
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 36)
        font_small = ImageFont.truetype("/Library/Fonts/SnellRoundhand.ttc", 28)
    except Exception:
        font = ImageFont.load_default()
        font_small = font

    draw.text((60, 60), "COMPREHENSIVE METABOLIC PANEL", fill="black", font=font)
    draw.text((60, 110), f"Specimen Date: 2026-0{rng.randint(1,9)}-{rng.randint(10,28)}", fill="black", font=font_small)

    top = 220
    row_h = 70
    col_x = [60, 500, 800, 1050, 1350]
    headers = ["Test", "Result", "Ref Low", "Ref High", "Unit"]
    for x, h in zip(col_x, headers):
        draw.text((x, top), h, fill="black", font=font)
    draw.line([(60, top + 50), (1640, top + 50)], fill="black", width=3)

    rows_html = []
    for i, (name, lo, hi, unit) in enumerate(tests):
        y = top + 70 + i * row_h
        result = str(round(rng.uniform(float(lo) * 0.85, float(hi) * 1.15), 1))
        row_vals = [name, result, lo, hi, unit]
        for x, v in zip(col_x, row_vals):
            draw.text((x, y), v, fill="black", font=font_small)
        rows_html.append(
            "<tr><td>{}</td><td>{}</td><td>{}</td><td>{}</td><td>{}</td></tr>".format(*row_vals)
        )

    img.save(path)

    table_html = (
        "<table>"
        "<tr><th>Test</th><th>Result</th><th>Ref Low</th><th>Ref High</th><th>Unit</th></tr>"
        + "".join(rows_html)
        + "</table>"
    )
    target = {
        "form_type": "basic_metabolic_panel",
        "fields": {"specimen_date": "REDACTED", "ordering_provider": "REDACTED"},
        "table_html": table_html,
    }
    return {"image": str(path), "target": target}


def build_synthetic_manifest(n_examples: int, out_dir: str) -> list[dict]:
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    manifest = []
    for i in range(n_examples):
        img_path = out_path / f"form_{i:04d}.png"
        record = _render_synthetic_lab_form(img_path, seed=i)
        manifest.append(record)
    return manifest


synthetic_manifest = build_synthetic_manifest(n_examples=24, out_dir="./synthetic_health_forms")
print(f"Built {len(synthetic_manifest)} synthetic (image, target) pairs.")
print(json.dumps(synthetic_manifest[0], indent=2)[:600], "...")
